In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
Mounted at /content/drive


In [2]:
import sys
sys.path.append("/content/drive/MyDrive/stocks/duet_regression")

from pipeline.config import DUETConfig
from pipeline import train, evaluate, predict
from pipeline.prepare_and_check import FinancialTimeSeriesPreparer
from duet.model import DUETModel
from torch.utils.data import DataLoader, TensorDataset
import torch
import joblib
import random
import numpy as np


In [8]:
# Настройки
joblib_path = "/content/drive/MyDrive/stocks/Data/DOGEUSDT/preprocessed/DOGEUSDT_1m_2024-01-01_to_2025-05-23_bb_20_2.joblib"
df = joblib.load(joblib_path)
device = "cuda" if torch.cuda.is_available() else "cpu"

config = DUETConfig(
    # =========================
    # Общие параметры
    # =========================
    timestamp_col = "timestamp",      # Название колонки содержащей timstamp (int)
    features = [
       'Open', 'High', 'Low', 'Close', 'SMA_20', 'dtrended_close', 'std_20', 'returns'
    ],
    forecast = [
        'std_20'
    ],            # Название колонок для предикта
    not_to_normalise = [],            # Название колонок, которые НЕ НАДО нормализовать
    scaler = 'MINMAX',                   # Тип нормализации (STD, MINMAX, QUANT)
    seq_len = 48,                     # Длина входной последовательности
    horizon = 6,                     # Длина выходного прогноза
    patch_len = 16,                   # Длина патча (TCM)
    stride = 8,                      # Шаг между патчами (TCM)
    moving_avg = 11,                  # Размер окна скользящего сглаживания

    # =========================
    # Параметры модели
    # =========================
    d_model = 128,                    # Размерность скрытого пространства в attention
    d_ff = 512,                      # Размерность feedforward слоя
    n_heads = 4,                      # Количество голов в multi-head attention
    e_layers = 3,                     # Количество слоев в encoder (CCM)
    dropout = 0.2,                    # Dropout во всех слоях attention
    fc_dropout = 0.2,                 # Dropout в выходном head слое
    activation = "relu",              # Активационная функция (relu, gelu, elu)
    num_experts = 2,                  # Число экспертов (в Router, если используется)

    # =========================
    # Режимы обработки
    # =========================
    CI = True,                 # Channel-Independent режим (если False — shared weights)
    use_router = False,          # Включить распределительный роутер
    timeenc = 0,                # Использовать time encoding (0 = без, 1 = sin/cos и т.п.)

    # =========================
    # Настройки обучения
    # =========================
    batch_size = 64,            # Размер батча
    epochs = 10,                # Количество эпох
    learning_rate = 0.0001,    # Скорость обучения
    weight_decay = 1e-6,      # L2 регуляризация
    patience = 20,               # Патенс для early stopping
    loss = "mse" ,              # Функция потерь: mse, mae, smape, mase
    metric = "mae",             # Основная метрика: mae, smape, mase

    # =========================
    # Прочее
    # =========================
    checkpoint_best = '/content/drive/MyDrive/stocks/duet_regression/weights/best_val_acc_weights_1min_w20.pt',       # Путь для сохранения лучших по val_accuracy весов
    checkpoint_final = '/content/drive/MyDrive/stocks/duet_regression/weights/final_weights_01.pt',      # Путь для сохранения финальных весов
    seed = 22221,                  # Фиксированное зерно генератора случайных чисел
    verbose = True,            # Печать хода обучения
)

"""
Устанавливает seed для numpy, random, torch (вкл. CUDA).
Гарантирует воспроизводимость.
"""

random.seed(config.seed)
np.random.seed(config.seed)
torch.manual_seed(config.seed)
torch.cuda.manual_seed_all(config.seed)


In [9]:

df['SMA_20'] = df['Close'].rolling(window=20).mean()
df['dtrended_close'] = df['Close'] - df['SMA_20']
df['std_20'] = df['Close'].rolling(window=20).std()
df['returns'] = df['Close'].pct_change()

In [10]:
# Загрузка данных
# df = joblib.load(joblib_path)
preparer = FinancialTimeSeriesPreparer(
    tz="UTC",
    timestamp_col=config.timestamp_col,
    drop_warmup=True,
)
df, _ = preparer.prepare(df, ensure_ohlcv=True)

# Обучение
model = DUETModel(config).to(device)

last_dim = len(set(config.features + config.forecast)) + (config.timeenc + 1) * 5
dummy_input = torch.tensor(np.zeros(shape = (config.batch_size, config.seq_len, last_dim )), dtype=torch.float32).to(device)

model(dummy_input)
# Загрузка весов
model.load_state_dict(torch.load(config.checkpoint_best)) # загрузка лучших весов
# model.load_state_dict(torch.load(config.checkpoint_final)) # загрузка финальных весов

model.eval()


Индекс уже в формате datetime
Инферированная частота: min
Пропусков в индексе не обнаружено
Окончательно удалено 19 строк содеражащих NaN


DUETModel(
  (tcm): TCM(
    (decomp): SeriesDecomposition(
      (avg_pool): AvgPool1d(kernel_size=(11,), stride=(1,), padding=(5,))
    )
    (extractor): LinearPatternExtractor(
      (projection): ModuleList(
        (0): Linear(in_features=32, out_features=128, bias=True)
      )
      (dropout): Dropout(p=0.2, inplace=False)
    )
  )
  (ccm): CCM(
    (enc_layers): ModuleList(
      (0-2): 3 x EncoderLayer(
        (attn): MultiHeadSelfAttention(
          (qkv_proj): Linear(in_features=128, out_features=384, bias=True)
          (out_proj): Linear(in_features=128, out_features=128, bias=True)
          (dropout): Dropout(p=0.2, inplace=False)
        )
        (dropout1): Dropout(p=0.2, inplace=False)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (activation): ReLU()
        (ffn): Sequential(
          (0): Linear(in_features=128, out_features=512, bias=True)
          (1): ReLU()
          (2): Dropout(p=0.2, inplace=False)
          (3): Line

In [11]:
# Предикт одного окна
from pipeline.predict import predict_window

#текущая точка t (sequence берется влево от t)
t = 100

x_window = df.iloc[t - config.seq_len + 1 : t + 1]
pred = predict_window(model, x_window, config, device=device)

print(pred)


[[6.1089821e-05]
 [5.7941292e-05]
 [5.5145912e-05]
 [5.3083877e-05]
 [5.2943615e-05]
 [5.4029708e-05]]


In [12]:
# Предикт целого датасета

from pipeline.predict import predict_dataset_batched

full_df = df

# Предикт в виде массива
pred = predict_dataset_batched(model, full_df, config, device=device)
print(f"Prediction array shape {pred.shape}")

Prediction array shape (732941, 6, 1)


In [13]:
# Предикт в виде датафрэйма
pred_df = predict.preds_array_to_dataframe(pred, full_df.index, config)
print(f"Prediction df shape {pred_df.shape}")
print(pred_df.columns)


Prediction df shape (732941, 6)
Index(['std_20_t+1', 'std_20_t+2', 'std_20_t+3', 'std_20_t+4', 'std_20_t+5',
       'std_20_t+6'],
      dtype='object')


In [15]:
import pandas as pd
df_with_preds= pd.merge(df, pred_df, how='outer', right_index=True, left_index=True)

In [16]:
df_with_preds.dropna(inplace=True)

In [17]:
df_with_preds

,timestamp,Open,High,Low,Close,Volume,sma,std,upper_band,lower_band,SMA_20,dtrended_close,std_20,returns,std_20_t+1,std_20_t+2,std_20_t+3,std_20_t+4,std_20_t+5,std_20_t+6
datetime,,,,,,,,,,,,,,,,,,,,
2024-01-01 01:06:00,1.704071e+12,0.08956,0.08961,0.08954,0.08960,184573.0,0.089762,0.000131,0.090024,0.089500,0.089762,-0.000162,0.000131,0.000335,0.000137,0.000140,0.000140,0.000141,0.000142,0.000141
2024-01-01 01:07:00,1.704071e+12,0.08960,0.08960,0.08956,0.08960,121308.0,0.089750,0.000135,0.090020,0.089481,0.089750,-0.000150,0.000135,0.000000,0.000138,0.000139,0.000139,0.000140,0.000140,0.000138
2024-01-01 01:08:00,1.704071e+12,0.08961,0.08961,0.08959,0.08960,553470.0,0.089737,0.000136,0.090008,0.089466,0.089737,-0.000137,0.000136,0.000000,0.000137,0.000137,0.000137,0.000136,0.000134,0.000130
2024-01-01 01:09:00,1.704071e+12,0.08959,0.08961,0.08958,0.08961,697080.0,0.089723,0.000134,0.089992,0.089455,0.089723,-0.000114,0.000134,0.000112,0.000136,0.000135,0.000134,0.000132,0.000128,0.000122
2024-01-01 01:10:00,1.704071e+12,0.08961,0.08972,0.08961,0.08966,278575.0,0.089713,0.000130,0.089973,0.089453,0.089713,-0.000053,0.000130,0.000558,0.000133,0.000131,0.000127,0.000124,0.000118,0.000111
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-05-23 23:55:00,1.748044e+12,0.22504,0.22504,0.22445,0.22462,999578.0,0.224534,0.000685,0.225905,0.223164,0.224534,0.000085,0.000685,-0.001866,0.000645,0.000627,0.000624,0.000635,0.000651,0.000663
2025-05-23 23:56:00,1.748045e+12,0.22463,0.22468,0.22418,0.22467,823690.0,0.224497,0.000654,0.225805,0.223189,0.224497,0.000173,0.000654,0.000223,0.000590,0.000572,0.000576,0.000596,0.000615,0.000625
2025-05-23 23:57:00,1.748045e+12,0.22467,0.22536,0.22452,0.22530,1002858.0,0.224506,0.000664,0.225833,0.223178,0.224506,0.000794,0.000664,0.002804,0.000666,0.000694,0.000724,0.000743,0.000731,0.000701


In [20]:
df_with_preds['bb_low_pred_1'] = df_with_preds['SMA_20'] - 2 * df_with_preds['std_20_t+1']
df_with_preds['bb_low_pred_2'] = df_with_preds['SMA_20'] - 2 * df_with_preds['std_20_t+2']
df_with_preds['bb_low_pred_3'] = df_with_preds['SMA_20'] - 2 * df_with_preds['std_20_t+3']
df_with_preds['bb_low_pred_4'] = df_with_preds['SMA_20'] - 2 * df_with_preds['std_20_t+4']
df_with_preds['bb_low_pred_5'] = df_with_preds['SMA_20'] - 2 * df_with_preds['std_20_t+5']
df_with_preds['bb_low_pred_6'] = df_with_preds['SMA_20'] - 2 * df_with_preds['std_20_t+6']

In [21]:
joblib.dump(df_with_preds, '/content/drive/MyDrive/stocks/Data/DOGEUSDT/preprocessed/DOGEUSDT_1m_2024-01-01_to_2025-05-23_std_20_predict.joblib')

['/content/drive/MyDrive/stocks/Data/DOGEUSDT/preprocessed/DOGEUSDT_1m_2024-01-01_to_2025-05-23_std_20_predict.joblib']

In [ ]:
def compute_row_slopes(matrix: np.ndarray, n: int) -> np.ndarray:
    """
    Для каждой строки массива (samples, columns) вычисляет угол наклона (slope)
    линейной регрессии, построенной по первым `n` колонкам.

    Параметры
    ----------
    matrix : np.ndarray
        Данные формы (samples, columns).
    n : int
        Сколько первых колонок брать для регрессии.
        n ≤ columns.

    Возврат
    -------
    slopes : np.ndarray
        Вектор из `samples` элементов — наклоны регрессий по каждой строке.
    """
    if n > matrix.shape[1]:
        raise ValueError(f"n ({n}) не может быть больше числа колонок ({matrix.shape[1]}).")

    # Берём только первые n колонок
    y = matrix[:, :n]                      # shape: (samples, n)
    x = np.arange(n, dtype=float)          # [0, 1, 2, ..., n-1]

    # Предварительные суммы для формулы МНК (vectorized)
    sum_x  = x.sum()                       # скаляр
    sum_x2 = (x**2).sum()                  # скаляр
    sum_y  = y.sum(axis=1)                 # shape: (samples,)
    sum_xy = (y * x).sum(axis=1)           # shape: (samples,)

    # Числитель и знаменатель формулы наклона
    denom = n * sum_x2 - sum_x**2          # скаляр, одинаков для всех строк
    slopes = (n * sum_xy - sum_x * sum_y) / denom

    return slopes

In [ ]:
er_15, er_30, er_60 = np.split(pred, 3, -1)


slope_er_15 = compute_row_slopes(er_15.squeeze(), 3)
slope_er_30 = compute_row_slopes(er_30.squeeze(), 3)
slope_er_60 = compute_row_slopes(er_60.squeeze(), 3)

print(slope_er_15.shape)
print(slope_er_30.shape)
print(slope_er_60.shape)

(526859,)
(526859,)
(526859,)


In [ ]:
df_with_preds['slope_er_15'] = slope_er_15
df_with_preds['slope_er_30'] = slope_er_30
df_with_preds['slope_er_60'] = slope_er_60

In [ ]:
df_with_preds.columns

Index(['Open', 'High', 'Low', 'Close', 'Volume', 'sigma_log_return_14',
       'sigma_bollinger_14', 'sigma_true_range_14', 'min_extr_distance_14',
       'max_extr_distance_14', 'efficiency_ratio_14', 'sigma_bolinger_15',
       'RSI_Close_15', 'er_15', 'sigma_bolinger_30', 'RSI_Close_30', 'er_30',
       'sigma_bolinger_60', 'RSI_Close_60', 'er_60', 'high_trend', 'low_trend',
       'high_trend_noise', 'low_trend_noise', 'signal_abs_speed',
       'signal_abs_accel', 'er_15_t+1', 'er_15_t+2', 'er_15_t+3', 'er_15_t+4',
       'er_15_t+5', 'er_15_t+6', 'er_30_t+1', 'er_30_t+2', 'er_30_t+3',
       'er_30_t+4', 'er_30_t+5', 'er_30_t+6', 'er_60_t+1', 'er_60_t+2',
       'er_60_t+3', 'er_60_t+4', 'er_60_t+5', 'er_60_t+6', 'slope_er_15',
       'slope_er_30', 'slope_er_60'],
      dtype='object')